In [16]:
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import cross_val_score, cross_validate
from sklearn.metrics import accuracy_score, classification_report
import wandb
import joblib
import pandas as pd
from modelos_utils import download_model_dfs

In [ ]:
config = {
            "tfidf_titulo_max_features": 2000,
            "tfidf_descripcion_max_features": 4000,
            "tfidf_tags_max_features": 2000,
            "tfidf_subtitulos_max_features": 5000,
            "ngram_range": (1,2),
            "svd_components": 300,
            "depth_values": range(1,100),
            'criterion': 'entropy',
            "cv_folds": 5
        }
preprocess = ColumnTransformer(
        transformers=[
            ("Titulo", TfidfVectorizer(max_features=config["tfidf_titulo_max_features"], ngram_range=config["ngram_range"]), "Titulo"),
            ("Descripcion", TfidfVectorizer(max_features=config["tfidf_descripcion_max_features"], ngram_range=config["ngram_range"]), "Descripcion"),
            ("Tags", TfidfVectorizer(max_features=config["tfidf_tags_max_features"], ngram_range=config["ngram_range"]), "Tags"),
            ("Subtitulos", TfidfVectorizer(max_features=config["tfidf_subtitulos_max_features"], ngram_range=config["ngram_range"]), "Subtitulos"),
            ("Generos", OneHotEncoder(), ["Generos"]),
            ("Duracion", StandardScaler(), ["Duracion"])
        ]
    )

In [3]:
best_n_leafs = None
best_acc = 0
best_model= None

In [4]:
df_train, df_validation, df_test = download_model_dfs()

In [8]:
df_train = pd.concat([df_train, df_validation])
X_train = df_train.drop(columns=["Rango_edad"])
y_train = df_train["Rango_edad"]
X_test = df_test.drop(columns=["Rango_edad"])
y_test = df_test["Rango_edad"]

In [9]:
results_table = pd.DataFrame(columns=['depth','cv_accuracy'])

In [19]:
for i in [1]:
    pipeline = Pipeline([
            ("preprocess", preprocess),
            ("svd", TruncatedSVD(n_components=config["svd_components"])),
            ("model", DecisionTreeClassifier(max_depth=i, criterion='entropy'))
        ])
    scores = cross_val_score(
            pipeline,
            X_train,
            y_train,
            cv=config["cv_folds"],
            scoring="accuracy",
            n_jobs=-1
        )

ValueError: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\alejo\OneDrive\Documentos\Uni\Segundo\PD\c2526-R1\.venv\Lib\site-packages\pandas\core\indexes\base.py", line 3641, in get_loc
    return self._engine.get_loc(casted_key)
           ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^
  File "pandas/_libs/index.pyx", line 168, in pandas._libs.index.IndexEngine.get_loc
  File "pandas/_libs/index.pyx", line 197, in pandas._libs.index.IndexEngine.get_loc
  File "pandas/_libs/hashtable_class_helper.pxi", line 7668, in pandas._libs.hashtable.PyObjectHashTable.get_item
  File "pandas/_libs/hashtable_class_helper.pxi", line 7676, in pandas._libs.hashtable.PyObjectHashTable.get_item
KeyError: 'Rango_eda